In [7]:
import numpy as np

# Simulation Configuration
NUM_PHOTONS = 1000                  # Number of photons Alice sends
CHANNEL_NOISE = 0.03                # 3% chance of a bit flip due to fiber noise
EVE_INTERCEPTION_PROBABILITY = 0.0  # Eve intercepts 50% of the photons

print(f"Simulation configured for {NUM_PHOTONS} photons.")
print(f"Channel Noise: {CHANNEL_NOISE*100}% | Eve Interception Prob: {EVE_INTERCEPTION_PROBABILITY*100}%")

Simulation configured for 1000 photons.
Channel Noise: 3.0% | Eve Interception Prob: 0.0%


In [8]:
# 0 = Rectilinear (+), 1 = Diagonal (x)
alice_bits = np.random.randint(0, 2, NUM_PHOTONS)
alice_bases = np.random.randint(0, 2, NUM_PHOTONS)

print("Alice's first 10 bits:  ", alice_bits[:10])
print("Alice's first 10 bases: ", alice_bases[:10])

Alice's first 10 bits:   [0 1 1 1 0 1 1 0 1 1]
Alice's first 10 bases:  [0 1 0 1 1 1 1 0 0 0]


In [9]:
# Copy Alice's photons as they enter the channel
channel_bits = alice_bits.copy()
channel_bases = alice_bases.copy()

# 1. EVE'S INTERCEPTION (Measure and Resend Attack)
for i in range(NUM_PHOTONS):
    if np.random.random() < EVE_INTERCEPTION_PROBABILITY:
        # Eve picks a random basis to measure Alice's photon
        eve_basis = np.random.randint(0, 2)

        if eve_basis != channel_bases[i]:
            # If bases don't match, Eve forces the bit to randomize (50/50 chance)
            channel_bits[i] = np.random.randint(0, 2)

        # Eve updates the photon state to match her measurement basis before sending to Bob
        channel_bases[i] = eve_basis

# 2. ENVIRONMENTAL NOISE (Bit Flips)
for i in range(NUM_PHOTONS):
    if np.random.random() < CHANNEL_NOISE:
        channel_bits[i] = 1 - channel_bits[i] # Flip the bit (0->1 or 1->0)

print("Photons have passed through the channel.")

Photons have passed through the channel.


In [10]:
bob_bases = np.random.randint(0, 2, NUM_PHOTONS)
bob_bits = np.empty(NUM_PHOTONS, dtype=int)

for i in range(NUM_PHOTONS):
    if bob_bases[i] == channel_bases[i]:
        # If Bob's basis matches the photon's current basis, he reads it perfectly
        bob_bits[i] = channel_bits[i]
    else:
        # If bases mismatch, Bob gets a random result (50/50)
        bob_bits[i] = np.random.randint(0, 2)

print("Bob's first 10 bits:  ", bob_bits[:10])
print("Bob's first 10 bases: ", bob_bases[:10])

Bob's first 10 bits:   [1 1 1 1 0 1 1 1 1 1]
Bob's first 10 bases:  [1 0 0 0 1 1 1 1 1 0]


In [11]:
# Find indices where Alice and Bob used the same basis
matching_bases_mask = (alice_bases == bob_bases)

# Sift keys
alice_sifted_key = alice_bits[matching_bases_mask]
bob_sifted_key = bob_bits[matching_bases_mask]

print(f"Photons with matching bases: {len(alice_sifted_key)} / {NUM_PHOTONS}")

Photons with matching bases: 509 / 1000


In [12]:
# Sample size for error checking (10% of the sifted key)
sample_size = max(1, len(alice_sifted_key) // 10)

sample_alice = alice_sifted_key[:sample_size]
sample_bob = bob_sifted_key[:sample_size]

# Calculate errors in sample
errors = np.sum(sample_alice != sample_bob)
qber = errors / sample_size

# Threshold for BB84 protocol security is typically ~11%
QBER_THRESHOLD = 0.11

print(f"--- QKD SECURITY REPORT ---")
print(f"Sifted Key Length: {len(alice_sifted_key)} bits")
print(f"Sample Size Checked: {sample_size} bits")
print(f"Errors Found: {errors}")
print(f"Calculated QBER: {qber * 100:.2f}%")

if qber > QBER_THRESHOLD:
    print(f"\n🛑 ALERT: QBER exceeds threshold ({QBER_THRESHOLD*100}%)!")
    print("Eavesdropping detected or channel too noisy. ABORTING KEY GENERATION.")
else:
    print(f"\n🟩 SUCCESS: QBER is within safe limits.")
    # Remove the sampled bits from the final secure key
    final_key_alice = alice_sifted_key[sample_size:]
    final_key_bob = bob_sifted_key[sample_size:]
    print(f"Final Secure Key Length: {len(final_key_alice)} bits")

--- QKD SECURITY REPORT ---
Sifted Key Length: 509 bits
Sample Size Checked: 50 bits
Errors Found: 1
Calculated QBER: 2.00%

🟩 SUCCESS: QBER is within safe limits.
Final Secure Key Length: 459 bits
